# CogAge All-Subject Dataset and Optimizer Comparison

This notebook loads the separately prepared subject datasets, displays their shapes, class counts, metadata, and sample sensor values, then performs the four-class optimizer assignment for subjects with all four activities in both original splits.

**Run from the workspace root** after running `python DATA_PREP/prepare_all_subjects.py`. The prepared files are read from the newest `DATA_PREP/processed_datasets_by_subject/run_*` folder. This notebook writes experiment results to a new `DATA_PREP/all_subjects_experiments/run_*` folder and does not replace the prepared datasets.

Tested dependencies: Python 3.12, NumPy, pandas, Matplotlib, scikit-learn, and joblib. Install with `python -m pip install numpy pandas matplotlib scikit-learn joblib jupyter ipykernel`.

**Data availability:** The four labels remain Bending=0, Walking=1, Sitting=2, Standing=3. In the inspected data, S1, S2, S5, and S7 have paired folders for all four classes. S3, S6, and S8 have only partial class sets, so they are displayed but excluded from four-class optimizer comparisons. S4 has no original `_2_extracted` folders and is skipped. This notebook preserves each subject’s `_1_extracted` train and `_2_extracted` test folders; validation Events are selected only from `_1_extracted`. Results from single-subject datasets are small and should not be presented as participant-independent generalization.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import time
import copy
import platform

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

WORKSPACE = Path.cwd()
if not (WORKSPACE / 'DATA_PREP').is_dir():
    raise FileNotFoundError('Run this notebook with the workspace root as the current folder.')
PREPARED_ROOT = WORKSPACE / 'DATA_PREP' / 'processed_datasets_by_subject'
run_folders = sorted(PREPARED_ROOT.glob('run_*'), key=lambda path: path.stat().st_mtime)
if not run_folders:
    raise FileNotFoundError('No prepared subject run found. Run python DATA_PREP/prepare_all_subjects.py first.')
PREPARED_RUN = run_folders[-1]
SUBJECT_RUN_SUMMARY = pd.read_csv(PREPARED_RUN / 'subject_preparation_summary.csv')
CLASS_MAPPING = json.loads((PREPARED_RUN / 'class_mapping.json').read_text(encoding='utf-8'))
CLASS_NAMES = {label: name for name, label in CLASS_MAPPING.items()}
CLASS_IDS = np.array(sorted(CLASS_NAMES), dtype=np.int64)
SUBJECT_DATASETS = {}

print(f'Prepared dataset run: {PREPARED_RUN}')
print('Preparation status by subject:')
display(SUBJECT_RUN_SUMMARY)
print('Folder-pair availability:')
display(pd.read_csv(PREPARED_RUN / 'subject_activity_inventory.csv'))

for _, row in SUBJECT_RUN_SUMMARY.iterrows():
    if row['status'] != 'prepared':
        continue
    subject = row['subject']
    folder = PREPARED_RUN / subject
    dataset = {
        'X_train': np.load(folder / 'X_train.npy'),
        'y_train': np.load(folder / 'y_train.npy'),
        'X_test': np.load(folder / 'X_test.npy'),
        'y_test': np.load(folder / 'y_test.npy'),
        'train_metadata': pd.read_csv(folder / 'train_metadata.csv'),
        'test_metadata': pd.read_csv(folder / 'test_metadata.csv'),
        'feature_names': json.loads((folder / 'feature_names.json').read_text(encoding='utf-8')),
        'class_mapping': json.loads((folder / 'class_mapping.json').read_text(encoding='utf-8')),
        'folder': folder,
    }
    SUBJECT_DATASETS[subject] = dataset
    print(f'\n{subject}: X_train={dataset["X_train"].shape}, y_train={dataset["y_train"].shape}, '
          f'X_test={dataset["X_test"].shape}, y_test={dataset["y_test"].shape}')
    print('Training label counts:', dict(Counter(dataset['y_train'].tolist())))
    print('Testing label counts: ', dict(Counter(dataset['y_test'].tolist())))

    # Show a small head: one row per window with label, provenance, and a few features.
    train_head = dataset['train_metadata'][['activity', 'label', 'original_folder', 'event', 'window_id']].head(5).copy()
    for feature_index, feature_name in enumerate(dataset['feature_names'][:5]):
        train_head[feature_name] = dataset['X_train'][:5, 0, feature_index]
    print(f'{subject} training dataset head (first timestep, first five features):')
    display(train_head)
    print(f'{subject} available class mapping: {dataset["class_mapping"]}')

FileNotFoundError: Run this notebook with the workspace root as the current folder.

In [ ]:
VALIDATION_FRACTION = 0.20
RANDOM_SEED = 42
MODEL_DATA = {}
EXCLUDED_FROM_MODEL = []

for subject, dataset in SUBJECT_DATASETS.items():
    y_train_subject = dataset['y_train']
    y_test_subject = dataset['y_test']
    train_classes = set(np.unique(y_train_subject).tolist())
    test_classes = set(np.unique(y_test_subject).tolist())
    if train_classes != set(CLASS_IDS.tolist()) or test_classes != set(CLASS_IDS.tolist()):
        missing_train = sorted(set(CLASS_IDS.tolist()) - train_classes)
        missing_test = sorted(set(CLASS_IDS.tolist()) - test_classes)
        EXCLUDED_FROM_MODEL.append({
            'subject': subject,
            'reason': f'missing train labels {missing_train}; missing test labels {missing_test}',
        })
        continue

    train_metadata = dataset['train_metadata'].reset_index(drop=True)
    event_groups = (
        train_metadata['original_folder'].astype(str) + '::' + train_metadata['event'].astype(str)
    ).to_numpy()
    train_indices = []
    validation_indices = []
    subject_number = int(subject[1:])
    rng = np.random.default_rng(RANDOM_SEED + subject_number)
    for class_id in CLASS_IDS:
        class_indices = np.flatnonzero(y_train_subject == class_id)
        class_groups = np.unique(event_groups[class_indices])
        if len(class_groups) < 2:
            raise ValueError(f'{subject} class {class_id} has fewer than two training Events.')
        validation_group_count = max(1, int(round(len(class_groups) * VALIDATION_FRACTION)))
        validation_group_count = min(validation_group_count, len(class_groups) - 1)
        held_out_groups = rng.choice(class_groups, size=validation_group_count, replace=False)
        goes_to_validation = np.isin(event_groups[class_indices], held_out_groups)
        validation_indices.extend(class_indices[goes_to_validation].tolist())
        train_indices.extend(class_indices[~goes_to_validation].tolist())

    train_indices = np.array(sorted(train_indices), dtype=int)
    validation_indices = np.array(sorted(validation_indices), dtype=int)
    assert set(event_groups[train_indices]).isdisjoint(set(event_groups[validation_indices]))

    X_original_train = dataset['X_train']
    X_original_test = dataset['X_test']
    # The saved per-subject scaler is a per-feature affine transform. Fitting this second
    # scaler on the model-training subset algebraically removes its earlier statistics.
    model_scaler = StandardScaler()
    model_scaler.fit(X_original_train[train_indices].reshape(-1, X_original_train.shape[-1]))

    def transform_subject_windows(windows, fitted_scaler):
        if not len(windows):
            return windows.astype(np.float32, copy=True)
        flat = windows.reshape(-1, windows.shape[-1])
        return fitted_scaler.transform(flat).reshape(windows.shape).astype(np.float32)

    X_fit_subject = transform_subject_windows(X_original_train[train_indices], model_scaler)
    X_valid_subject = transform_subject_windows(X_original_train[validation_indices], model_scaler)
    X_test_subject = transform_subject_windows(X_original_test, model_scaler)
    subject_data = {
        'X_train': X_fit_subject.reshape(len(X_fit_subject), -1).astype(np.float64),
        'y_train': y_train_subject[train_indices].astype(np.int64),
        'X_validation': X_valid_subject.reshape(len(X_valid_subject), -1).astype(np.float64),
        'y_validation': y_train_subject[validation_indices].astype(np.int64),
        'X_test': X_test_subject.reshape(len(X_test_subject), -1).astype(np.float64),
        'y_test': y_test_subject.astype(np.int64),
        'scaler': model_scaler,
        'train_metadata': train_metadata.iloc[train_indices].reset_index(drop=True),
        'validation_metadata': train_metadata.iloc[validation_indices].reset_index(drop=True),
        'test_metadata': dataset['test_metadata'].reset_index(drop=True),
        'feature_names': dataset['feature_names'],
    }
    MODEL_DATA[subject] = subject_data
    assert subject_data['train_metadata']['original_folder'].str.endswith('_1_extracted').all()
    assert subject_data['validation_metadata']['original_folder'].str.endswith('_1_extracted').all()
    assert subject_data['test_metadata']['original_folder'].str.endswith('_2_extracted').all()

split_rows = []
for subject, data in MODEL_DATA.items():
    for class_id in CLASS_IDS:
        split_rows.append({
            'subject': subject,
            'activity': CLASS_NAMES[int(class_id)],
            'label': int(class_id),
            'model_train': int(np.sum(data['y_train'] == class_id)),
            'validation': int(np.sum(data['y_validation'] == class_id)),
            'held_out_test': int(np.sum(data['y_test'] == class_id)),
        })
SPLIT_COUNTS = pd.DataFrame(split_rows)
print('Four-class subjects eligible for optimizer runs:', sorted(MODEL_DATA))
print('Subjects excluded from 4-class experiments (still inspected above):')
display(pd.DataFrame(EXCLUDED_FROM_MODEL))
print('Event-grouped train/validation/test counts:')
display(SPLIT_COUNTS)
print('Limitation: each per-subject dataset has a single participant; this is not participant-independent evaluation.')

## Fixed network and eight optimizers

The model uses a `4,224 -> 32 -> 16 -> 4` fully connected architecture with ReLU hidden layers and four raw logits (135,796 parameters). Loss is sparse categorical cross-entropy from logits. Each optimizer gets the same initialization, 100-epoch limit, base learning rate 0.01, mini-batch size 8, and data partitions. Test results use the best-validation-loss checkpoint; the original test set is not used for selection.

In [ ]:
EXPERIMENT_SEED = 42
MAX_EPOCHS = 100
MINI_BATCH_SIZE = 8
LEARNING_RATE = 0.01
MOMENTUM = 0.9
ADAMW_WEIGHT_DECAY = 0.01
HIDDEN_DIMS = (32, 16)
OUTPUT_DIM = len(CLASS_IDS)
INPUT_DIM = next(iter(MODEL_DATA.values()))['X_train'].shape[1]
PARAMETER_COUNT = (INPUT_DIM + 1) * HIDDEN_DIMS[0] + (HIDDEN_DIMS[0] + 1) * HIDDEN_DIMS[1] + (HIDDEN_DIMS[1] + 1) * OUTPUT_DIM

OPTIMIZERS = [
    {'name': 'Full-batch GD', 'key': 'full_batch_gd', 'batch_size': None},
    {'name': 'Mini-batch GD', 'key': 'mini_batch_gd', 'batch_size': MINI_BATCH_SIZE},
    {'name': 'Stochastic GD', 'key': 'sgd', 'batch_size': 1},
    {'name': 'SGD + Momentum', 'key': 'momentum', 'batch_size': MINI_BATCH_SIZE},
    {'name': 'Adagrad', 'key': 'adagrad', 'batch_size': MINI_BATCH_SIZE},
    {'name': 'RMSprop', 'key': 'rmsprop', 'batch_size': MINI_BATCH_SIZE},
    {'name': 'Adam', 'key': 'adam', 'batch_size': MINI_BATCH_SIZE},
    {'name': 'AdamW', 'key': 'adamw', 'batch_size': MINI_BATCH_SIZE},
]


def initialize_model(input_dim, seed):
    rng = np.random.default_rng(seed)
    dimensions = (input_dim, *HIDDEN_DIMS, OUTPUT_DIM)
    params = {}
    for layer in range(1, len(dimensions)):
        fan_in, fan_out = dimensions[layer - 1], dimensions[layer]
        params[f'W{layer}'] = rng.normal(0.0, np.sqrt(2.0 / fan_in), (fan_in, fan_out))
        params[f'b{layer}'] = np.zeros(fan_out, dtype=np.float64)
    return params


def forward_pass(features, params):
    z1 = features @ params['W1'] + params['b1']
    h1 = np.maximum(z1, 0)
    z2 = h1 @ params['W2'] + params['b2']
    h2 = np.maximum(z2, 0)
    logits = h2 @ params['W3'] + params['b3']
    return logits, (features, z1, h1, z2, h2)


def softmax(logits):
    shifted = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def evaluate_model(features, labels, params):
    logits, _ = forward_pass(features, params)
    probabilities = softmax(logits)
    row_indices = np.arange(len(labels))
    loss = -np.log(np.clip(probabilities[row_indices, labels], 1e-12, 1.0)).mean()
    accuracy = np.mean(np.argmax(logits, axis=1) == labels)
    return float(loss), float(accuracy), np.argmax(logits, axis=1)


def calculate_gradients(features, labels, params):
    logits, cache = forward_pass(features, params)
    batch_features, z1, h1, z2, h2 = cache
    probabilities = softmax(logits)
    row_indices = np.arange(len(labels))
    loss = -np.log(np.clip(probabilities[row_indices, labels], 1e-12, 1.0)).mean()
    dlogits = probabilities.copy()
    dlogits[row_indices, labels] -= 1.0
    dlogits /= len(labels)
    gradients = {
        'W3': h2.T @ dlogits,
        'b3': dlogits.sum(axis=0),
    }
    dz2 = (dlogits @ params['W3'].T) * (z2 > 0)
    gradients['W2'] = h1.T @ dz2
    gradients['b2'] = dz2.sum(axis=0)
    dz1 = (dz2 @ params['W2'].T) * (z1 > 0)
    gradients['W1'] = batch_features.T @ dz1
    gradients['b1'] = dz1.sum(axis=0)
    return float(loss), gradients


def create_optimizer_state(key, params):
    zeros = {name: np.zeros_like(value) for name, value in params.items()}
    if key == 'momentum':
        return {'velocity': zeros}
    if key == 'adagrad':
        return {'sum_squares': zeros}
    if key == 'rmsprop':
        return {'average_squares': zeros}
    if key in ('adam', 'adamw'):
        return {'first': copy.deepcopy(zeros), 'second': copy.deepcopy(zeros)}
    return {}


def optimizer_step(params, gradients, state, key, step):
    epsilon = 1e-8
    for name, parameter in params.items():
        gradient = gradients[name]
        if key in ('full_batch_gd', 'mini_batch_gd', 'sgd'):
            parameter -= LEARNING_RATE * gradient
        elif key == 'momentum':
            state['velocity'][name] = MOMENTUM * state['velocity'][name] + gradient
            parameter -= LEARNING_RATE * state['velocity'][name]
        elif key == 'adagrad':
            state['sum_squares'][name] += gradient * gradient
            parameter -= LEARNING_RATE * gradient / (np.sqrt(state['sum_squares'][name]) + epsilon)
        elif key == 'rmsprop':
            alpha = 0.99
            state['average_squares'][name] = alpha * state['average_squares'][name] + (1 - alpha) * gradient * gradient
            parameter -= LEARNING_RATE * gradient / (np.sqrt(state['average_squares'][name]) + epsilon)
        elif key in ('adam', 'adamw'):
            beta1, beta2 = 0.9, 0.999
            first = state['first'][name]
            second = state['second'][name]
            first *= beta1
            first += (1 - beta1) * gradient
            second *= beta2
            second += (1 - beta2) * gradient * gradient
            corrected_first = first / (1 - beta1 ** step)
            corrected_second = second / (1 - beta2 ** step)
            if key == 'adamw':
                parameter *= 1 - LEARNING_RATE * ADAMW_WEIGHT_DECAY
            parameter -= LEARNING_RATE * corrected_first / (np.sqrt(corrected_second) + epsilon)
        else:
            raise ValueError(f'Unknown optimizer {key}')


EXPERIMENT_ROOT = WORKSPACE / 'DATA_PREP' / 'all_subjects_experiments'
EXPERIMENT_DIR = EXPERIMENT_ROOT / f'run_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=False)
for subject in MODEL_DATA:
    (EXPERIMENT_DIR / subject / 'models').mkdir(parents=True, exist_ok=False)

experiment_histories = {}
experiment_confusions = {}
experiment_results_rows = []

for subject, subject_data in MODEL_DATA.items():
    features_train = subject_data['X_train']
    labels_train = subject_data['y_train']
    features_validation = subject_data['X_validation']
    labels_validation = subject_data['y_validation']
    features_test = subject_data['X_test']
    labels_test = subject_data['y_test']
    print(f'\n{subject}: train={len(labels_train)}, validation={len(labels_validation)}, test={len(labels_test)}')

    for optimizer in OPTIMIZERS:
        name, key = optimizer['name'], optimizer['key']
        batch_size = optimizer['batch_size']
        params = initialize_model(INPUT_DIM, EXPERIMENT_SEED)
        state = create_optimizer_state(key, params)
        shuffle_rng = np.random.default_rng(EXPERIMENT_SEED + 1)
        updates_per_epoch = 1 if batch_size is None else int(np.ceil(len(features_train) / batch_size))
        total_updates = 0
        best_validation_loss = np.inf
        best_epoch = 0
        best_params = copy.deepcopy(params)
        history_rows = []
        started = time.perf_counter()

        for epoch in range(1, MAX_EPOCHS + 1):
            if batch_size is None:
                batches = [np.arange(len(features_train))]
            else:
                permutation = shuffle_rng.permutation(len(features_train))
                batches = [permutation[start:start + batch_size] for start in range(0, len(permutation), batch_size)]

            for indices in batches:
                _, gradients = calculate_gradients(features_train[indices], labels_train[indices], params)
                total_updates += 1
                optimizer_step(params, gradients, state, key, total_updates)

            train_loss, train_accuracy, _ = evaluate_model(features_train, labels_train, params)
            validation_loss, validation_accuracy, _ = evaluate_model(features_validation, labels_validation, params)
            if validation_loss < best_validation_loss:
                best_validation_loss = validation_loss
                best_epoch = epoch
                best_params = copy.deepcopy(params)
            history_rows.append({
                'subject': subject,
                'optimizer': name,
                'epoch': epoch,
                'train_loss': train_loss,
                'validation_loss': validation_loss,
                'train_accuracy': train_accuracy,
                'validation_accuracy': validation_accuracy,
                'updates_this_epoch': updates_per_epoch,
                'cumulative_updates': total_updates,
                'elapsed_seconds': time.perf_counter() - started,
            })

        training_seconds = time.perf_counter() - started
        # The held-out test set is scored only after validation has selected the checkpoint.
        test_loss, test_accuracy, test_predictions = evaluate_model(features_test, labels_test, best_params)
        test_macro_f1 = f1_score(labels_test, test_predictions, labels=CLASS_IDS, average='macro', zero_division=0)
        matrix = confusion_matrix(labels_test, test_predictions, labels=CLASS_IDS)
        experiment_histories[(subject, name)] = pd.DataFrame(history_rows)
        experiment_confusions[(subject, name)] = matrix
        np.savez_compressed(EXPERIMENT_DIR / subject / 'models' / f'{key}_best.npz', **best_params)
        experiment_results_rows.append({
            'subject': subject,
            'optimizer': name,
            'best_validation_loss': best_validation_loss,
            'best_epoch': best_epoch,
            'test_loss': test_loss,
            'test_accuracy': test_accuracy,
            'test_macro_f1': float(test_macro_f1),
            'training_seconds': training_seconds,
            'updates_per_epoch': updates_per_epoch,
            'total_updates': total_updates,
        })
        print(f'  {name}: val_loss={best_validation_loss:.4f} at epoch {best_epoch}; '
              f'test_acc={test_accuracy:.3f}; macro_F1={test_macro_f1:.3f}; updates={total_updates}')

experiment_results = pd.DataFrame(experiment_results_rows)
experiment_history = pd.concat(experiment_histories.values(), ignore_index=True)
experiment_results.to_csv(EXPERIMENT_DIR / 'optimizer_summary.csv', index=False)
experiment_history.to_csv(EXPERIMENT_DIR / 'training_history.csv', index=False)
(EXPERIMENT_DIR / 'confusion_matrices.json').write_text(
    json.dumps({f'{subject}|{optimizer}': matrix.tolist()
                for (subject, optimizer), matrix in experiment_confusions.items()}, indent=2),
    encoding='utf-8',
)
with (EXPERIMENT_DIR / 'experiment_config.json').open('w', encoding='utf-8') as config_file:
    json.dump({
        'subjects': list(MODEL_DATA),
        'classes': CLASS_MAPPING,
        'seed': EXPERIMENT_SEED,
        'epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'mini_batch_size': MINI_BATCH_SIZE,
        'momentum': MOMENTUM,
        'adamw_weight_decay': ADAMW_WEIGHT_DECAY,
        'architecture': [INPUT_DIM, *HIDDEN_DIMS, OUTPUT_DIM],
        'parameter_count': PARAMETER_COUNT,
        'validation_fraction': VALIDATION_FRACTION,
        'validation_grouping': 'source recording folder plus Event, within _1_extracted only',
        'test_selection': 'best validation-loss checkpoint; test evaluated once per run',
    }, config_file, indent=2)

display(experiment_results.sort_values(['subject', 'best_validation_loss']).reset_index(drop=True))
print(f'Experiment artifacts: {EXPERIMENT_DIR.resolve()}')

In [ ]:
from IPython.display import Image, display

FIGURE_ROOT = EXPERIMENT_DIR / 'figures'
FIGURE_ROOT.mkdir(parents=True, exist_ok=False)
optimizer_names = [item['name'] for item in OPTIMIZERS]

# Assignment EDA figure 1: original train/test window counts by class and subject.
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for axis, (subject, dataset) in zip(axes.flat, SUBJECT_DATASETS.items()):
    train_counts = Counter(dataset['y_train'].tolist())
    test_counts = Counter(dataset['y_test'].tolist())
    x = np.arange(len(CLASS_IDS))
    width = 0.38
    axis.bar(x - width / 2, [train_counts[int(c)] for c in CLASS_IDS], width, label='Train')
    axis.bar(x + width / 2, [test_counts[int(c)] for c in CLASS_IDS], width, label='Test')
    axis.set_xticks(x, [CLASS_NAMES[int(c)] for c in CLASS_IDS], rotation=20, ha='right')
    axis.set_title(subject)
    axis.set_ylabel('128-step windows')
    axis.grid(axis='y', alpha=0.2)
axes[0, 0].legend()
fig.suptitle('Exploratory class counts by subject and original split')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'class_counts_by_subject.png', dpi=170, bbox_inches='tight')
plt.show()

# Assignment EDA figure 2: one representative PhoneAccelerometer feature by class.
reference_subject = next((sid for sid in ('S1', 'S2', 'S5', 'S7') if sid in SUBJECT_DATASETS), None)
if reference_subject is not None:
    reference = SUBJECT_DATASETS[reference_subject]
    feature_index = next(
        (i for i, name in enumerate(reference['feature_names']) if 'PhoneAccelerometer' in name), 0
    )
    fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
    for axis, class_id in zip(axes, CLASS_IDS):
        sample_indices = np.flatnonzero(reference['y_train'] == class_id)
        if len(sample_indices):
            axis.plot(
                np.arange(128) * 20,
                reference['X_train'][sample_indices[0], :, feature_index],
                linewidth=0.9,
            )
            axis.set_ylabel(CLASS_NAMES[int(class_id)])
        else:
            axis.text(0.5, 0.5, 'No training sample', ha='center', va='center', transform=axis.transAxes)
        axis.grid(alpha=0.2)
    axes[-1].set_xlabel('Time within 128-step window (ms)')
    fig.suptitle(f'{reference_subject} representative training signal: {reference["feature_names"][feature_index]} (scaled)')
    fig.tight_layout()
    fig.savefig(FIGURE_ROOT / 'representative_sensor_by_class.png', dpi=170, bbox_inches='tight')
    plt.show()

# Per-subject optimizer curves and per-run 4 x 4 test confusion matrices.
for subject in MODEL_DATA:
    subject_figure_dir = FIGURE_ROOT / subject
    subject_figure_dir.mkdir(parents=True, exist_ok=False)

    fig, axes = plt.subplots(4, 2, figsize=(13, 15), sharex=True)
    for axis, optimizer_name in zip(axes.flat, optimizer_names):
        history = experiment_histories[(subject, optimizer_name)]
        axis.plot(history['epoch'], history['train_loss'], label='Train')
        axis.plot(history['epoch'], history['validation_loss'], label='Validation')
        axis.set_title(optimizer_name)
        axis.set_ylabel('Cross-entropy')
        axis.grid(alpha=0.2)
    axes[0, 0].legend()
    axes[-1, 0].set_xlabel('Epoch')
    axes[-1, 1].set_xlabel('Epoch')
    fig.suptitle(f'{subject}: training and validation loss')
    fig.tight_layout()
    fig.savefig(subject_figure_dir / 'loss_by_epoch.png', dpi=160, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(4, 2, figsize=(13, 15), sharex=True, sharey=True)
    for axis, optimizer_name in zip(axes.flat, optimizer_names):
        history = experiment_histories[(subject, optimizer_name)]
        axis.plot(history['epoch'], history['train_accuracy'], label='Train')
        axis.plot(history['epoch'], history['validation_accuracy'], label='Validation')
        axis.set_title(optimizer_name)
        axis.set_ylabel('Accuracy')
        axis.set_ylim(0, 1.02)
        axis.grid(alpha=0.2)
    axes[0, 0].legend(loc='lower right')
    axes[-1, 0].set_xlabel('Epoch')
    axes[-1, 1].set_xlabel('Epoch')
    fig.suptitle(f'{subject}: training and validation accuracy')
    fig.tight_layout()
    fig.savefig(subject_figure_dir / 'accuracy_by_epoch.png', dpi=160, bbox_inches='tight')
    plt.show()

    fig, axis = plt.subplots(figsize=(9, 5))
    for optimizer_name in optimizer_names:
        history = experiment_histories[(subject, optimizer_name)]
        axis.plot(history['cumulative_updates'], history['validation_loss'], label=optimizer_name)
    axis.set_xscale('log')
    axis.set_yscale('log')
    axis.set_xlabel('Cumulative updates (log scale)')
    axis.set_ylabel('Validation loss (log scale)')
    axis.set_title(f'{subject}: validation convergence by optimizer updates')
    axis.grid(alpha=0.2, which='both')
    axis.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(subject_figure_dir / 'validation_loss_by_updates.png', dpi=160, bbox_inches='tight')
    plt.show()

    subject_results = experiment_results[experiment_results['subject'] == subject].set_index('optimizer').loc[optimizer_names].reset_index()
    positions = np.arange(len(optimizer_names))
    bar_width = 0.38
    fig, axis = plt.subplots(figsize=(11, 5))
    axis.bar(positions - bar_width / 2, subject_results['test_accuracy'], bar_width, label='Test accuracy')
    axis.bar(positions + bar_width / 2, subject_results['test_macro_f1'], bar_width, label='Test macro F1')
    axis.set_xticks(positions, optimizer_names, rotation=30, ha='right')
    axis.set_ylim(0, 1.05)
    axis.set_ylabel('Score')
    axis.set_title(f'{subject}: held-out test metrics')
    axis.legend()
    axis.grid(axis='y', alpha=0.2)
    fig.tight_layout()
    fig.savefig(subject_figure_dir / 'test_metrics_by_optimizer.png', dpi=160, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(4, 2, figsize=(13, 15))
    for axis, optimizer_name in zip(axes.flat, optimizer_names):
        matrix = experiment_confusions[(subject, optimizer_name)]
        image = axis.imshow(matrix, cmap='Blues', vmin=0, vmax=max(1, matrix.max()))
        axis.set_title(optimizer_name)
        axis.set_xticks(range(4), [CLASS_NAMES[int(c)] for c in CLASS_IDS], rotation=35, ha='right', fontsize=8)
        axis.set_yticks(range(4), [CLASS_NAMES[int(c)] for c in CLASS_IDS], fontsize=8)
        axis.set_xlabel('Predicted')
        axis.set_ylabel('True')
        for row in range(4):
            for column in range(4):
                axis.text(column, row, str(matrix[row, column]), ha='center', va='center', fontsize=9,
                          color='white' if matrix[row, column] > matrix.max() / 2 else 'black')
    fig.colorbar(image, ax=axes.ravel().tolist(), shrink=0.78, label='Test windows')
    fig.suptitle(f'{subject}: held-out test confusion matrices (rows=true, columns=predicted)')
    fig.savefig(subject_figure_dir / 'confusion_matrices.png', dpi=170, bbox_inches='tight')
    plt.show()

print('Saved EDA and subject-wise comparison figures under:', FIGURE_ROOT.resolve())

In [ ]:
# Extend class-count visualization to every prepared subject, including partial class sets.
subject_names = list(SUBJECT_DATASETS)
fig, axes = plt.subplots(4, 2, figsize=(13, 15), sharey=True)
for axis, subject in zip(axes.flat, subject_names):
    dataset = SUBJECT_DATASETS[subject]
    train_counts = Counter(dataset['y_train'].tolist())
    test_counts = Counter(dataset['y_test'].tolist())
    positions = np.arange(len(CLASS_IDS))
    width = 0.38
    axis.bar(positions - width / 2, [train_counts[int(c)] for c in CLASS_IDS], width, label='Train')
    axis.bar(positions + width / 2, [test_counts[int(c)] for c in CLASS_IDS], width, label='Test')
    axis.set_xticks(positions, [CLASS_NAMES[int(c)] for c in CLASS_IDS], rotation=25, ha='right', fontsize=8)
    axis.set_title(subject)
    axis.set_ylabel('Windows')
    axis.grid(axis='y', alpha=0.2)
for axis in axes.flat[len(subject_names):]:
    axis.set_visible(False)
axes.flat[0].legend()
fig.suptitle('Train/test window counts for every prepared subject, including absent classes')
fig.tight_layout()
fig.savefig(FIGURE_ROOT / 'class_counts_all_prepared_subjects.png', dpi=170, bbox_inches='tight')
plt.show()

In [ ]:
import textwrap
from matplotlib.backends.backend_pdf import PdfPages


def save_text_page(pdf, title, sections):
    page = plt.figure(figsize=(8.27, 11.69))
    page.text(0.07, 0.96, title, fontsize=17, weight='bold', va='top')
    y = 0.91
    for heading, text in sections:
        page.text(0.08, y, heading, fontsize=11, weight='bold', va='top')
        y -= 0.033
        wrapped = textwrap.fill(text, width=95)
        page.text(0.08, y, wrapped, fontsize=8.6, va='top', linespacing=1.3)
        y -= 0.022 * (wrapped.count('\n') + 1) + 0.03
    pdf.savefig(page, bbox_inches='tight')
    plt.close(page)


def add_plot_page(pdf, title, figure_path, caption):
    page = plt.figure(figsize=(8.27, 11.69))
    page.text(0.07, 0.96, title, fontsize=16, weight='bold', va='top')
    axis = page.add_axes([0.05, 0.14, 0.90, 0.78])
    axis.imshow(plt.imread(figure_path))
    axis.set_aspect('auto')
    axis.axis('off')
    page.text(0.07, 0.08, textwrap.fill(caption, width=110), fontsize=8.5, va='bottom')
    pdf.savefig(page, bbox_inches='tight')
    plt.close(page)


# Build one six-page report; detailed per-run CSV/JSON and figures remain alongside it.
report_path = EXPERIMENT_DIR / 'CogAge_all_subjects_optimizer_report.pdf'
subject_overview = []
for subject, data in MODEL_DATA.items():
    per_class_train = ', '.join(
        f'{CLASS_NAMES[int(class_id)]}:{int(np.sum(data["y_train"] == class_id))}' for class_id in CLASS_IDS
    )
    per_class_validation = ', '.join(
        f'{CLASS_NAMES[int(class_id)]}:{int(np.sum(data["y_validation"] == class_id))}' for class_id in CLASS_IDS
    )
    per_class_test = ', '.join(
        f'{CLASS_NAMES[int(class_id)]}:{int(np.sum(data["y_test"] == class_id))}' for class_id in CLASS_IDS
    )
    subject_overview.append(
        f'{subject}: model train {len(data["y_train"])} [{per_class_train}]; '
        f'validation {len(data["y_validation"])} [{per_class_validation}]; '
        f'test {len(data["y_test"])} [{per_class_test}]'
    )

with PdfPages(report_path) as pdf:
    page = plt.figure(figsize=(8.27, 11.69))
    page.text(0.07, 0.96, 'CogAge multi-subject optimizer comparison', fontsize=17, weight='bold', va='top')
    page.text(0.08, 0.91, 'Dataset, fixed model, and split policy', fontsize=11, weight='bold', va='top')
    overview = (
        'Four labels are held fixed across subjects: Bending=0, Walking=1, Sitting=2, Standing=3. '
        'Only subjects with all four activities in both original splits are included in four-class experiments. '
        'The inspected prepared datasets include all S1-S8: S1, S2, S5, S7 are complete; S3, S6, S8 are partial; '
        'S4 has no original test folders. Partial subjects are previewed but not assigned fabricated classes.'
    )
    page.text(0.08, 0.875, textwrap.fill(overview, width=103), fontsize=8.5, va='top', linespacing=1.3)
    split_text = (
        'Original _1_extracted folders supply training windows; _2_extracted folders supply held-out testing windows. '
        'A seeded 20% Event-group validation split is made within each class from the original training folders only. '
        'No Event crosses model-train and validation. The model scaler is refitted on model-train windows and then applied '
        'to validation and test. Test metrics use the best validation-loss checkpoint and are not used for tuning.'
    )
    page.text(0.08, 0.79, textwrap.fill(split_text, width=103), fontsize=8.5, va='top', linespacing=1.3)
    architecture = (
        f'Input: 128 x 33 flattened ({INPUT_DIM}); network {INPUT_DIM} -> 32 -> 16 -> 4 logits; '
        f'ReLU hidden layers; {PARAMETER_COUNT:,} parameters; stable sparse cross-entropy from logits. '
        f'NumPy implementation; He-normal initialization seed {EXPERIMENT_SEED}; {MAX_EPOCHS} epochs; '
        f'learning rate {LEARNING_RATE}; mini-batch {MINI_BATCH_SIZE}; momentum {MOMENTUM}; '
        f'AdamW decoupled weight decay {ADAMW_WEIGHT_DECAY}. Full-batch, mini-batch and stochastic GD update '
        'once per full set, mini batch and example respectively; the five adaptive/momentum configurations use mini batches.'
    )
    page.text(0.08, 0.70, textwrap.fill(architecture, width=103), fontsize=8.5, va='top', linespacing=1.3)
    page.text(0.08, 0.60, 'Per-subject class and split counts', fontsize=11, weight='bold', va='top')
    page.text(0.08, 0.565, textwrap.fill(' | '.join(subject_overview), width=108), fontsize=7.6, va='top', linespacing=1.25)

    results_for_report = experiment_results.copy()
    results_for_report['best_validation_loss'] = results_for_report['best_validation_loss'].map(lambda value: f'{value:.3f}')
    results_for_report['test_accuracy'] = results_for_report['test_accuracy'].map(lambda value: f'{value:.3f}')
    results_for_report['test_macro_f1'] = results_for_report['test_macro_f1'].map(lambda value: f'{value:.3f}')
    results_for_report['training_seconds'] = results_for_report['training_seconds'].map(lambda value: f'{value:.2f}')
    results_for_report['total_updates'] = results_for_report['total_updates'].map(lambda value: f'{int(value):,}')
    result_table = results_for_report[
        ['subject', 'optimizer', 'best_validation_loss', 'best_epoch', 'test_accuracy', 'test_macro_f1', 'training_seconds', 'total_updates']
    ]
    table_axis = page.add_axes([0.055, 0.04, 0.90, 0.39])
    table_axis.axis('off')
    table = table_axis.table(
        cellText=result_table.values.tolist(),
        colLabels=['Subject', 'Optimizer', 'Best val loss', 'Epoch', 'Test acc.', 'Macro F1', 'Seconds', 'Updates'],
        cellLoc='center', colLoc='center', loc='center',
        colWidths=[0.08, 0.19, 0.13, 0.07, 0.10, 0.10, 0.10, 0.12],
    )
    table.auto_set_font_size(False)
    table.set_fontsize(5.8)
    table.scale(1, 1.20)
    pdf.savefig(page, bbox_inches='tight')
    plt.close(page)

    loss_fig = plt.figure(figsize=(12, 9))
    loss_axes = loss_fig.subplots(2, 2)
    for axis, subject in zip(loss_axes.flat, MODEL_DATA):
        for optimizer_name in optimizer_names:
            history = experiment_histories[(subject, optimizer_name)]
            axis.plot(history['epoch'], history['validation_loss'], label=optimizer_name, linewidth=0.9)
        axis.set_title(f'{subject}: validation loss')
        axis.set_xlabel('Epoch')
        axis.set_ylabel('Cross-entropy')
        axis.grid(alpha=0.2)
    loss_axes.flat[0].legend(fontsize=6, ncol=2)
    loss_fig.tight_layout()
    loss_path = FIGURE_ROOT / 'report_validation_loss_all_subjects.png'
    loss_fig.savefig(loss_path, dpi=170, bbox_inches='tight')
    plt.close(loss_fig)
    add_plot_page(pdf, 'Validation loss by epoch', loss_path,
                  'Figure 1. Validation loss for all eight optimizer runs, shown separately for each complete subject. Checkpoints are selected using validation loss.')

    accuracy_fig = plt.figure(figsize=(12, 9))
    accuracy_axes = accuracy_fig.subplots(2, 2)
    for axis, subject in zip(accuracy_axes.flat, MODEL_DATA):
        for optimizer_name in optimizer_names:
            history = experiment_histories[(subject, optimizer_name)]
            axis.plot(history['epoch'], history['validation_accuracy'], label=optimizer_name, linewidth=0.9)
        axis.set_title(f'{subject}: validation accuracy')
        axis.set_xlabel('Epoch')
        axis.set_ylabel('Accuracy')
        axis.set_ylim(0, 1.02)
        axis.grid(alpha=0.2)
    accuracy_axes.flat[0].legend(fontsize=6, ncol=2, loc='lower right')
    accuracy_fig.tight_layout()
    accuracy_path = FIGURE_ROOT / 'report_validation_accuracy_all_subjects.png'
    accuracy_fig.savefig(accuracy_path, dpi=170, bbox_inches='tight')
    plt.close(accuracy_fig)
    add_plot_page(pdf, 'Validation accuracy by epoch', accuracy_path,
                  'Figure 2. Validation accuracy for each optimizer and subject. Validation contains few Events, so a single misclassification changes accuracy substantially.')

    compare_fig, compare_axes = plt.subplots(4, 2, figsize=(13, 15))
    for axis, subject in zip(compare_axes.flat, MODEL_DATA):
        for optimizer_name in optimizer_names:
            history = experiment_histories[(subject, optimizer_name)]
            axis.plot(history['cumulative_updates'], history['validation_loss'], label=optimizer_name, linewidth=0.9)
        axis.set_xscale('log')
        axis.set_yscale('log')
        axis.set_title(f'{subject}: validation loss vs updates')
        axis.set_xlabel('Cumulative updates (log)')
        axis.set_ylabel('Validation loss (log)')
        axis.grid(alpha=0.2, which='both')
    compare_axes[0, 0].legend(fontsize=6, ncol=2)
    for axis, subject in zip(compare_axes[:, 1], MODEL_DATA):
        subject_results = experiment_results[experiment_results['subject'] == subject].set_index('optimizer').loc[optimizer_names]
        x = np.arange(len(optimizer_names))
        axis.bar(x - 0.19, subject_results['test_accuracy'], 0.38, label='Accuracy')
        axis.bar(x + 0.19, subject_results['test_macro_f1'], 0.38, label='Macro F1')
        axis.set_title(f'{subject}: held-out test metrics')
        axis.set_xticks(x, optimizer_names, rotation=55, ha='right', fontsize=7)
        axis.set_ylim(0, 1.05)
        axis.grid(axis='y', alpha=0.2)
    compare_axes[0, 1].legend(fontsize=7)
    compare_fig.suptitle('Update-based convergence and held-out test scores')
    compare_fig.tight_layout()
    compare_path = FIGURE_ROOT / 'report_convergence_and_test_metrics.png'
    compare_fig.savefig(compare_path, dpi=170, bbox_inches='tight')
    plt.close(compare_fig)
    add_plot_page(pdf, 'Convergence and test comparisons', compare_path,
                  'Figure 3. Left panels compare validation loss against cumulative updates; right panels show test accuracy and macro F1. Epochs alone are not comparable because batch sizes produce different update counts.')

    eligible_subjects = list(MODEL_DATA)
    for start in range(0, len(eligible_subjects), 2):
        subject_pair = eligible_subjects[start:start + 2]
        figure, axes = plt.subplots(4, 4, figsize=(11.69, 8.27))
        axes = np.asarray(axes)
        for subject_row, subject in enumerate(subject_pair):
            for optimizer_index, optimizer_name in enumerate(optimizer_names):
                row_offset = subject_row * 2
                matrix = experiment_confusions[(subject, optimizer_name)]
                image = axes[row_offset, optimizer_index // 4].imshow(
                    matrix, cmap='Blues', vmin=0, vmax=max(1, matrix.max())
                )
                # The second row of panels carries the remaining four optimizers.
                target_axis = axes[row_offset + optimizer_index // 4, optimizer_index % 4]
                target_axis.clear()
                target_axis.imshow(matrix, cmap='Blues', vmin=0, vmax=max(1, matrix.max()))
                target_axis.set_title(f'{subject} | {optimizer_name}', fontsize=7)
                target_axis.set_xticks(range(4), [CLASS_NAMES[int(c)] for c in CLASS_IDS], rotation=45, ha='right', fontsize=5)
                target_axis.set_yticks(range(4), [CLASS_NAMES[int(c)] for c in CLASS_IDS], fontsize=5)
                for row in range(4):
                    for column in range(4):
                        target_axis.text(column, row, str(matrix[row, column]), ha='center', va='center', fontsize=7,
                                         color='white' if matrix[row, column] > matrix.max() / 2 else 'black')
        figure.suptitle(f'Held-out test confusion matrices: {", ".join(subject_pair)}')
        figure.tight_layout()
        confusion_path = FIGURE_ROOT / f'report_confusion_matrices_{"_".join(subject_pair)}.png'
        figure.savefig(confusion_path, dpi=190, bbox_inches='tight')
        plt.close(figure)
        add_plot_page(pdf, f'Confusion matrices: {", ".join(subject_pair)}', confusion_path,
                      'Figure 4. One 4 x 4 test confusion matrix per optimizer. Rows are true labels and columns are predicted labels; class order is Bending, Walking, Sitting, Standing.')

    top_rows = []
    for subject in MODEL_DATA:
        best = experiment_results[experiment_results['subject'] == subject].sort_values('test_macro_f1', ascending=False).iloc[0]
        top_rows.append(f"{subject}: top test macro F1 was {best['test_macro_f1']:.3f} ({best['optimizer']}); "
                        f"test accuracy {best['test_accuracy']:.3f}. This is descriptive, not a tuning rule.")
    add_text_page(pdf, 'Interpretation, reproducibility, and limitations', [
        ('Observed results', ' | '.join(top_rows)),
        ('Update-frequency comparison', 'Full-batch GD performs one update per epoch; mini-batch configurations perform ceil(N/8) updates; stochastic GD performs one update per example. Compare validation loss against updates and runtime rather than treating equal epochs as equal work.'),
        ('Optimizer notes', 'Momentum accumulates a fraction of the previous update. Adagrad accumulates squared gradients and typically reduces effective learning rates over time. RMSprop uses a decaying average of squared gradients. Adam combines first and second moments; AdamW applies weight decay separately from the adaptive gradient update.'),
        ('Limitations and cleaning', 'Single-subject experiments are not participant-independent. Subjects differ in available classes and have few Events, making validation estimates noisy. No class balancing is performed. Per-subject raw audits, synchronization/interpolation counts, discarded windows, and class counts are available in each prepared subject folder. Results should not be generalized beyond these recordings.'),
        ('Reproducibility and disclosure', f'Run python DATA_PREP/prepare_all_subjects.py, then execute this notebook top to bottom. Random seed {EXPERIMENT_SEED}; NumPy implementation; experiment artifacts are stored in a fresh timestamped folder. GitHub Copilot assisted with drafting and debugging code; verify and disclose assistance according to course policy.'),
    ])

print('Saved report:', report_path.resolve())
print('Experiment figures and per-run matrices:', FIGURE_ROOT.resolve())